# Figure 3 — DIA-NN Class I phosphosite pipeline (Spectronaut-matched)

Processes the DIA-NN outputs of the HeLa sorted-cell dilution series and counts **Class I
phosphosites** (localization probability ≥ 0.75) using a collapse definition **identical** to the
Spectronaut path, for a fair head-to-head comparison. Counts are reported **with multiplicity**
(`gene_site_multiplicity`) and **with multiplicity collapsed** (`gene_site` — matches Figure 3a).

## Filtering (matches manuscript §8.2)
1. Drop entries without gene information.
2. `PG.Q.Value ≤ 0.05`, `Global.PG.Q.Value ≤ 0.01`, `Lib.PG.Q.Value ≤ 0.01` (MBR on),
   `Quantity.Quality ≥ 0.5`, `PG.MaxLFQ.Quality ≥ 0.7`.
3. Keep only phosphopeptides (`Phospho`, UniMod:21).
4. Collapse per site and per multiplicity, then filter on **per-(site, run) localization ≥ 0.75**.

## Why this is a bulletproof comparison
Both engines are reduced to the **same counting unit** and counted gene-level with an identical
per-run Class I rule:

| step | Spectronaut | DIA-NN |
|---|---|---|
| collapse | `core.process_ptm_site_report` (Spectronaut PTM Site Report) | `alphaphos.PeptideCollapse` (Hogrebe) on the DIA-NN report adapted to Spectronaut PSM schema |
| site key | `{PG}~{Gene}_{aa}{pos}_M{mult}` | **same format** |
| per-site localization | `PTM.SiteProbability` per run | `Site.Occupancy.Probabilities` per run |
| Class I | quant present **and** loc ≥ 0.75, per run | **same rule** |
| count | unique `{Gene}_{aa}{pos}_M{mult}` (or `{Gene}_{aa}{pos}` collapsed) per run | **same unified counter** |

The collapsed counter reproduces Figure 3a's `count_sites_per_sample_ptm_report` numbers to within
a handful of sites (<0.03 %).


## 1. Setup

In [8]:
import sys, re, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- alphaphos PeptideCollapse, loaded standalone (avoids the package __init__ heavy deps) ---
ALPHAPHOS_COLLAPSE = r"D:\Projects\alphaPhos\src\alphaphos\preprocess\collapse.py"
_spec = importlib.util.spec_from_file_location("alphaphos_collapse", ALPHAPHOS_COLLAPSE)
apc = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(apc)
PeptideCollapse = apc.PeptideCollapse

# --- Spectronaut Class I path (paper's existing pipeline) ---
sys.path.insert(0, r"D:\Projects\nanoPhos_env\src")
from core import process_ptm_site_report

DIANN_DIR   = Path(r"Z:\Denys_nanoPhos\PRIDE\analysis_data\revision\figure3\diann\figure3")
SPECTRO_DIR = Path(r"D:\Projects\nanoPhos_env\raw_data\revision\figure3")
FIG_DIR     = Path(r"D:\Projects\nanoPhos\figures_upd\figure3")

CELL_COUNTS = [100, 300, 500, 1000, 2000, 3000]
CUTOFF      = 0.75
MBR_ON      = True   # Lib.PG.Q.Value filter only applied when MBR is on
print("PeptideCollapse + core.process_ptm_site_report loaded.")

PeptideCollapse + core.process_ptm_site_report loaded.


## 2. DIA-NN → Spectronaut-PSM adapter

DIA-NN columns are mapped onto the exact PSM schema `PeptideCollapse` consumes, so the
**identical** collapse code runs on both engines.

- `Modified.Sequence` → `EG.PrecursorId` (`_SEQ[Phospho (STY)]..._.charge`)
- `Site.Occupancy.Probabilities` → `EG.PTMLocalizationProbabilities` (per-site loc, Spectronaut format)
- `Protein.Sites` (S/T/Y only; carbamidomethyl-C dropped) + within-peptide phospho position →
  `PEP.PeptidePosition`
- `PTM.Site.Confidence` → `EG.PTMAssayProbability`; `Precursor.Quantity` → `EG.TotalQuantity (Settings)`;
  `Run` → `R.FileName`; `Genes`/`Protein.Group` → `PG.Genes`/`PG.ProteinGroups`

In [9]:
_UNIMOD = re.compile(r"\(UniMod:\d+\)")

def diann_precursor_id(modseq, charge):
    """DIA-NN Modified.Sequence -> Spectronaut EG.PrecursorId."""
    s = modseq.replace("(UniMod:21)", "[Phospho (STY)]")
    return "_" + _UNIMOD.sub("", s) + "_." + str(int(charge))

def diann_loc_probs(occ):
    """DIA-NN Site.Occupancy.Probabilities -> Spectronaut EG.PTMLocalizationProbabilities.
    'AAS(UniMod:21){1.000000}LPT{0.848000}K2' -> '_AAS[Phospho (STY): 100%]LPT[Phospho (STY): 84.8%]K_'"""
    if not isinstance(occ, str):
        return np.nan
    s = re.sub(r"\d+$", "", _UNIMOD.sub("", occ))            # drop UniMod tags + trailing charge digit
    s = re.sub(r"\{([\d.]+)\}",
               lambda m: "[Phospho (STY): %.4g%%]" % (float(m.group(1)) * 100), s)
    return "_" + s + "_"

def _first_phospho_abs(protein_sites):
    """First phospho (S/T/Y) absolute position from DIA-NN Protein.Sites, ignoring
    carbamidomethyl-C entries DIA-NN bundles in (e.g. '[P35221:C116,S118]')."""
    inner = str(protein_sites).strip("[]").split(";")[0]     # leading protein
    if ":" not in inner:
        return np.nan
    sty = [t for t in inner.split(":", 1)[1].split(",") if t[:1] in "STY"]
    return int(re.sub(r"\D", "", sty[0])) if sty else np.nan

def diann_pep_start(modseq, protein_sites):
    within = len(_UNIMOD.sub("", modseq[:modseq.index("(UniMod:21)")]))
    a = _first_phospho_abs(protein_sites)
    return int(a) - within + 1 if a == a else np.nan

def diann_to_psm(df):
    a = pd.DataFrame({
        "R.FileName":                     df["Run"].astype(str),
        "EG.PrecursorId":                 [diann_precursor_id(m, c)
                                           for m, c in zip(df["Modified.Sequence"], df["Precursor.Charge"])],
        "EG.TotalQuantity (Settings)":    df["Precursor.Quantity"].values,
        "PEP.PeptidePosition":            [diann_pep_start(m, p)
                                           for m, p in zip(df["Modified.Sequence"], df["Protein.Sites"])],
        "EG.PTMAssayProbability":         df["PTM.Site.Confidence"].values,
        "EG.PTMLocalizationProbabilities":[diann_loc_probs(o) for o in df["Site.Occupancy.Probabilities"]],
        "PG.Genes":                       df["Genes"].astype(str).str.split(";").str[0],
        "PG.ProteinGroups":               df["Protein.Group"].astype(str),
    })
    n_unmappable = int(a["PEP.PeptidePosition"].isna().sum())
    a = a.dropna(subset=["PEP.PeptidePosition"]).copy()
    a["PEP.PeptidePosition"] = a["PEP.PeptidePosition"].astype(int)
    return a, n_unmappable

## 3. DIA-NN load + §8.2 QC filtering

In [10]:
def load_diann_filtered(parquet_path, mbr=MBR_ON, verbose=True):
    df = pd.read_parquet(parquet_path)
    funnel = {"raw_precursors": len(df)}
    df = df[df["Genes"].notna() & (df["Genes"].astype(str).str.strip() != "")]
    funnel["has_gene"] = len(df)
    flt = ((df["PG.Q.Value"]        <= 0.05) &
           (df["Global.PG.Q.Value"] <= 0.01) &
           (df["Quantity.Quality"]  >= 0.5) &
           (df["PG.MaxLFQ.Quality"] >= 0.7))
    if mbr:
        flt &= (df["Lib.PG.Q.Value"] <= 0.01)
    df = df[flt];                                       funnel["after_qc"]    = len(df)
    df = df[df["Modified.Sequence"].str.contains("UniMod:21", na=False)].copy()
    funnel["phospho"] = len(df)
    df = df[df["Protein.Sites"].notna() & df["Site.Occupancy.Probabilities"].notna()]
    funnel["localizable"] = len(df)
    if verbose:
        print("  funnel:", " -> ".join(f"{k}={v:,}" for k, v in funnel.items()))
    return df, funnel


def diann_classI(parquet_path, cutoff=CUTOFF, verbose=True):
    """filter -> adapt -> PeptideCollapse (per-run, same site key as Spectronaut)."""
    df, funnel = load_diann_filtered(parquet_path, verbose=verbose)
    psm, n_unmappable = diann_to_psm(df)
    pc = PeptideCollapse(verbose=False)
    pc.process_complete_pipeline(
        psm, cutoff=cutoff, collapse_level="PG", aggregation_method="sum",
        return_both=False, add_kinase_sequences=False, noise_floor_filter=True,
        localization_strategy="per_run",
    )
    diag = {"funnel": funnel, "unmappable_pep_pos": n_unmappable,
            "loc_fallback_pct": pc.processing_stats["per_site_localization_fallback_pct"],
            "site_keys": len(pc.site_data)}
    if verbose:
        print(f"  collapsed site keys={diag['site_keys']:,} | unmappable={n_unmappable} | "
              f"per-site loc fallback={diag['loc_fallback_pct']}%")
    return pc.site_data, diag

## 4. Unified Class I counter (identical for both engines; multiplicity toggle)

Both engines emit `PTM_Collapse_key = {ProteinGroup}~{Gene}_{aa}{pos}_M{mult}` and mask the
per-(site, run) quant cell to `NaN` when localization < 0.75. One counter applied to both
guarantees an identical definition. `collapse_multiplicity=True` drops the `_M{mult}` suffix →
unique `{Gene}_{aa}{pos}` (the Figure 3a convention).

In [11]:
_META = {"Protein_group", "Gene_group", "PTM_0_aa", "PTM_pos", "PTM_mult123", "PTM_flank",
         "PTM_Collapse_key", "UPD_seq", "PTM_localization", "kinase_sequence",
         "Protein_Collapse_key", "PG.Genes", "PG.ProteinGroups"}

def _sample_cols(site_data):
    return [c for c in site_data.columns
            if c not in _META and pd.api.types.is_numeric_dtype(site_data[c])]

def _site_key(site_data, collapse_multiplicity=False):
    """{Gene}_{aa}{pos}_M{mult}, or {Gene}_{aa}{pos} when multiplicity collapsed."""
    key = site_data["PTM_Collapse_key"].str.split("~", n=1).str[1]
    if collapse_multiplicity:
        key = key.str.replace(r"_M\d+$", "", regex=True)
    return key

def per_run_classI(site_data, collapse_multiplicity=False):
    sc = _sample_cols(site_data)
    key = _site_key(site_data, collapse_multiplicity)
    return site_data[sc].notna().assign(_k=key.values).groupby("_k").any().sum(axis=0)

def classI_site_set(site_data, collapse_multiplicity=False):
    sc = _sample_cols(site_data)
    key = _site_key(site_data, collapse_multiplicity)
    present_any = site_data[sc].notna().any(axis=1).values
    return set(key[present_any])

## 5. Run both engines across the dilution series (both multiplicity modes)

In [12]:
def find_diann_parquet(n):
    # main report only; file naming varies ("hela_report_100.parquet" vs "report_hela_sorted_3000.parquet")
    cands = [p for p in DIANN_DIR.glob(f"hela_{n}_cells/*.parquet")
             if p.name.endswith(f"_{n}.parquet")
             and "first-pass" not in p.name and "site_report" not in p.name]
    if not cands:
        raise FileNotFoundError(f"No main DIA-NN report for {n} cells in {DIANN_DIR}")
    return cands[0]

def find_spectronaut_tsv(n):
    cands = list(SPECTRO_DIR.glob(f"*nanoPhos_new_HeLa_sorted_{n}cells_Report.tsv"))
    if not cands:
        raise FileNotFoundError(f"No Spectronaut report for {n} cells in {SPECTRO_DIR}")
    return cands[0]

MODES = [("with_mult", False), ("no_mult", True)]   # no_mult matches Figure 3a

rows, diags = [], {}
sets = {m: {"diann": {}, "spectro": {}} for m, _ in MODES}
for n in CELL_COUNTS:
    print(f"\n=== {n} cells ===")
    print("[DIA-NN]");      d_sd, diag = diann_classI(find_diann_parquet(n)); diags[n] = diag
    print("[Spectronaut]"); s_sd = process_ptm_site_report(
        pd.read_csv(find_spectronaut_tsv(n), sep="\t", low_memory=False), cutoff=CUTOFF)["site_data"]

    row = {"cells": n, "loc_fallback_pct": diag["loc_fallback_pct"], "unmappable": diag["unmappable_pep_pos"]}
    for mode, cm in MODES:
        d_pr = per_run_classI(d_sd, cm); s_pr = per_run_classI(s_sd, cm)
        sets[mode]["diann"][n]   = classI_site_set(d_sd, cm)
        sets[mode]["spectro"][n] = classI_site_set(s_sd, cm)
        row[f"DIANN_{mode}"]       = round(float(d_pr.mean()), 1)
        row[f"Spectronaut_{mode}"] = round(float(s_pr.mean()), 1)
        row[f"D/S%_{mode}"]        = round(100 * d_pr.mean() / s_pr.mean(), 1)
        row[f"DIANN_{mode}_runs"]       = [int(x) for x in d_pr.round()]
        row[f"Spectronaut_{mode}_runs"] = [int(x) for x in s_pr.round()]
    rows.append(row)

summary = pd.DataFrame(rows)
summary[["cells", "DIANN_no_mult", "Spectronaut_no_mult", "D/S%_no_mult",
         "DIANN_with_mult", "Spectronaut_with_mult", "D/S%_with_mult",
         "loc_fallback_pct", "unmappable"]]


=== 100 cells ===
[DIA-NN]


INFO	PeptideCollapse:collapse.py:process_complete_pipeline()- Starting complete pipeline
INFO	PeptideCollapse:collapse.py:load_data()- Loaded 11563 rows
INFO	PeptideCollapse:collapse.py:load_data()- Total rows loaded: 11563
INFO	PeptideCollapse:collapse.py:load_data()- Unique samples (R.FileName): 3
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing: 11563 rows remaining after phospho filter, 0 non-phospho removed


  funnel: raw_precursors=27,288 -> has_gene=27,076 -> after_qc=25,692 -> phospho=11,563 -> localizable=11,563


INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing complete: 11563 phospho rows retained
INFO	PeptideCollapse:collapse.py:collapse_to_sites()- Starting site-level collapse (aggregation=sum, cutoff=0.75, strategy=per_run)
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Site collapse preprocessing: 11563 rows remaining, 11563 phospho, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Per-site localization: 12722 rows got per-site prob, 0 fell back to joint prob (0.0% fallback)
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Site pivot: 5191 keys x 3 samples, NaN count: 2851, zero count: 0
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Per-(site, run) localization matrix built: 4027 sites x 3 runs (82.7% non-NaN)
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Site aggregation: 4027 collapse keys, NaN count in aggregated matrix: 0
INFO	PeptideCollapse:collapse.py:_create_sit

  collapsed site keys=3,071 | unmappable=0 | per-site loc fallback=0.0%
[Spectronaut]
Dropped 2,625 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 8,040 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 4,189 → 4,171.
Final: 4,171 sites × 3 samples.

=== 300 cells ===
[DIA-NN]


...

INFO	PeptideCollapse:collapse.py:process_complete_pipeline()- Starting complete pipeline
INFO	PeptideCollapse:collapse.py:load_data()- Loaded 68018 rows
INFO	PeptideCollapse:collapse.py:load_data()- Total rows loaded: 68018
INFO	PeptideCollapse:collapse.py:load_data()- Unique samples (R.FileName): 3
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing: 68018 rows remaining after phospho filter, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing complete: 68018 phospho rows retained
INFO	PeptideCollapse:collapse.py:collapse_to_sites()- Starting site-level collapse (aggregation=sum, cutoff=0.75, strategy=per_run)
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Site collapse preprocessing: 68018 rows remaining, 68018 phospho, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Per-site localization: 79797 rows got per-site prob, 0 fell back to joint prob (0.0% fallback)
INFO	PeptideCollapse

  collapsed site keys=16,783 | unmappable=0 | per-site loc fallback=0.0%
[Spectronaut]
Dropped 5,731 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 44,374 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 20,965 → 20,899.
Final: 20,899 sites × 3 samples.

=== 3000 cells ===
[DIA-NN]
  funnel: raw_precursors=103,178 -> has_gene=103,130 -> after_qc=98,114 -> phospho=76,875 -> localizable=76,875


INFO	PeptideCollapse:collapse.py:process_complete_pipeline()- Starting complete pipeline
INFO	PeptideCollapse:collapse.py:load_data()- Loaded 76875 rows
INFO	PeptideCollapse:collapse.py:load_data()- Total rows loaded: 76875
INFO	PeptideCollapse:collapse.py:load_data()- Unique samples (R.FileName): 3
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing: 76875 rows remaining after phospho filter, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing complete: 76875 phospho rows retained
INFO	PeptideCollapse:collapse.py:collapse_to_sites()- Starting site-level collapse (aggregation=sum, cutoff=0.75, strategy=per_run)
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Site collapse preprocessing: 76875 rows remaining, 76875 phospho, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Per-site localization: 90233 rows got per-site prob, 3 fell back to joint prob (0.0% fallback)
INFO	PeptideCollapse

  collapsed site keys=18,511 | unmappable=0 | per-site loc fallback=0.0%
[Spectronaut]
Dropped 6,083 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 50,740 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 23,797 → 23,726.
Final: 23,726 sites × 3 samples.


,cells,DIANN_no_mult,Spectronaut_no_mult,D/S%_no_mult,DIANN_with_mult,Spectronaut_with_mult,D/S%_with_mult,loc_fallback_pct,unmappable
0,100,2136.7,2593.7,82.4,2275.0,2666.7,85.3,0.0,0
1,300,5195.3,6240.0,83.3,5689.0,6507.3,87.4,0.0,0
2,500,6770.0,8234.3,82.2,7491.0,8660.3,86.5,0.0,0
3,1000,9005.3,10408.7,86.5,10173.7,11081.7,91.8,0.0,0
4,2000,12089.7,13707.3,88.2,13824.0,14746.7,93.7,0.0,0
5,3000,13521.0,15623.7,86.5,15480.7,16862.7,91.8,0.0,0


## 6. Site-level overlap (both multiplicity modes)

In [13]:
ov = []
for n in CELL_COUNTS:
    rec = {"cells": n}
    for mode, _ in MODES:
        d, s = sets[mode]["diann"][n], sets[mode]["spectro"][n]
        inter = len(d & s); union = len(d | s)
        rec[f"DIANN_{mode}"] = len(d); rec[f"Spectronaut_{mode}"] = len(s)
        rec[f"shared_{mode}"] = inter
        rec[f"%Spec_in_DIANN_{mode}"] = round(100 * inter / max(1, len(s)), 1)
        rec[f"Jaccard_{mode}"] = round(100 * inter / max(1, union), 1)
    ov.append(rec)
overlap = pd.DataFrame(ov)
overlap

,cells,DIANN_with_mult,Spectronaut_with_mult,shared_with_mult,%Spec_in_DIANN_with_mult,Jaccard_with_mult,DIANN_no_mult,Spectronaut_no_mult,shared_no_mult,%Spec_in_DIANN_no_mult,Jaccard_no_mult
0,100,3071,4171,2018,48.4,38.6,2864,4064,1977,48.6,39.9
1,300,7516,10060,5166,51.4,41.6,6825,9625,4974,51.7,43.3
2,500,9864,13174,6890,52.3,42.7,8866,12527,6607,52.7,44.7
3,1000,12561,16160,8870,54.9,44.7,11044,15186,8383,55.2,47.0
4,2000,16783,20894,11879,56.9,46.0,14578,19445,11129,57.2,48.6
5,3000,18511,23724,13278,56.0,45.9,16081,21998,12399,56.4,48.3


In [21]:
# Overlap vs sorted cells (Reviewer 3, comment 2). no_mult = Figure 3a convention.
x = [str(n) for n in overlap["cells"]]
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=overlap["%Spec_in_DIANN_no_mult"], mode="lines+markers",
    name="% Spectronaut sites confirmed by DIA-NN",
    marker=dict(size=11, color="#2b8cbe", line=dict(width=0.5, color="black")), line=dict(width=2, color="#2b8cbe")))
fig.add_trace(go.Scatter(x=x, y=overlap["Jaccard_no_mult"], mode="lines+markers", name="Jaccard overlap",
    marker=dict(size=11, color="#6a51a3", line=dict(width=0.5, color="black")), line=dict(width=2, color="#6a51a3", dash="dash")))
fig.update_layout(width=660, height=600, template="plotly_white",
    xaxis_title="Sorted HeLa cells", yaxis_title="DIA-NN ↔ Spectronaut overlap (%)",
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.7)"))
fig.update_yaxes(range=[0, 70])
fig.show()
fig.write_image(str(FIG_DIR / "figure3_diann_spectronaut_overlap.pdf"), width=600, height=600)
#fig.write_image(str(FIG_DIR / "figure3_diann_spectronaut_overlap.png"), width=660, height=600, scale=2)

## 7. Figures — Class I depth vs cells (DIA-NN vs Spectronaut)

In [19]:
DIANN_COLOR, SPEC_COLOR = "#2b8cbe", "darkred"
x = [str(n) for n in summary["cells"]]
FIG_DIR.mkdir(parents=True, exist_ok=True)

def depth_fig(mode, title):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=summary[f"Spectronaut_{mode}"], mode="lines+markers", name="Spectronaut",
        marker=dict(size=12, color=SPEC_COLOR, line=dict(width=0.5, color="black")), line=dict(width=2, color=SPEC_COLOR)))
    fig.add_trace(go.Scatter(x=x, y=summary[f"DIANN_{mode}"], mode="lines+markers", name="DIA-NN",
        marker=dict(size=12, color=DIANN_COLOR, line=dict(width=0.5, color="black")), line=dict(width=2, color=DIANN_COLOR)))
    fig.update_layout(width=640, height=600, template="plotly_white", title=title,
        xaxis_title="Sorted HeLa cells", yaxis_title="Class I phosphosites (loc ≥ 0.75, per run)",
        legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.7)"))
    return fig

# no-multiplicity (matches Figure 3a) — the primary comparison
fig_nm = depth_fig("no_mult", "Class I sites (multiplicity collapsed; matches Fig. 3a)")
fig_nm.show()
fig_nm.write_image(str(FIG_DIR / "figure3_diann_vs_spectronaut_classI_noMult.pdf"), width=600, height=600)
#fig_nm.write_image(str(FIG_DIR / "figure3_diann_vs_spectronaut_classI_noMult.png"), width=640, height=600, scale=2)

# with-multiplicity
fig_wm = depth_fig("with_mult", "Class I sites (with multiplicity)")
fig_wm.show()
#fig_wm.write_image(str(FIG_DIR / "figure3_diann_vs_spectronaut_classI_withMult.pdf"), width=640, height=600)
#fig_wm.write_image(str(FIG_DIR / "figure3_diann_vs_spectronaut_classI_withMult.png"), width=640, height=600, scale=2)

## 8. Export tables

In [ ]:
out = FIG_DIR / "figure3_diann_vs_spectronaut_classI.xlsx"
_str = lambda df: df.assign(**{c: df[c].astype(str) for c in df.columns if df[c].apply(lambda v: isinstance(v, list)).any()})
with pd.ExcelWriter(out) as xl:
    _str(summary).to_excel(xl, sheet_name="per_run_counts", index=False)
    overlap.to_excel(xl, sheet_name="site_overlap", index=False)
print("saved", out)
print("\nQC: per-site localization fallback should be ~0% and unmappable 0:")
print(summary[["cells", "loc_fallback_pct", "unmappable"]].to_string(index=False))

## 9. Are engine-divergent sites low-abundance? (Reviewer 3, comment 2)
Sites detected by only one engine vs sites shared by both, compared on per-site
intensity. Backs the statement that engine-unique Class I sites are predominantly
low-abundance (and therefore drop below one engine's detection threshold at lower input).
Localization is *not* the driver — Spectronaut-only sites are confidently localized.


In [16]:
from scipy import stats

def per_site_intensity(site_data):
    """Per gene_site (multiplicity collapsed, Fig 3a unit) max mean-log2 intensity."""
    k = _site_key(site_data, collapse_multiplicity=True)
    cols = _sample_cols(site_data)
    inten = site_data[cols].mean(axis=1)                 # mean log2 across reps (NaN-skipped)
    return pd.DataFrame({"k": k.values, "i": inten.values}).groupby("k")["i"].max()

abund_rows, site_data_cache = [], {}
for n in CELL_COUNTS:
    d_sd, _ = diann_classI(find_diann_parquet(n), verbose=False)
    s_sd = process_ptm_site_report(
        pd.read_csv(find_spectronaut_tsv(n), sep="\t", low_memory=False), cutoff=CUTOFF)["site_data"]
    site_data_cache[n] = (s_sd, d_sd)                    # reused by the figure cell

    Si, Di = per_site_intensity(s_sd), per_site_intensity(d_sd)
    S, D = set(Si.index), set(Di.index)
    shared, s_only, d_only = S & D, S - D, D - S

    sh_s, on_s = Si[list(shared)].dropna(), Si[list(s_only)].dropna()
    sh_d, on_d = Di[list(shared)].dropna(), Di[list(d_only)].dropna()
    p_s = stats.mannwhitneyu(sh_s, on_s, alternative="greater").pvalue
    p_d = stats.mannwhitneyu(sh_d, on_d, alternative="greater").pvalue
    abund_rows.append({
        "cells": n,
        "Spec_shared_med":  round(float(sh_s.median()), 2),
        "Spec_only_med":    round(float(on_s.median()), 2),
        "Spec_p":           p_s,
        "DIANN_shared_med": round(float(sh_d.median()), 2),
        "DIANN_only_med":   round(float(on_d.median()), 2),
        "DIANN_p":          p_d,
    })

abundance = pd.DataFrame(abund_rows)
abundance


INFO	PeptideCollapse:collapse.py:process_complete_pipeline()- Starting complete pipeline
INFO	PeptideCollapse:collapse.py:load_data()- Loaded 11563 rows
INFO	PeptideCollapse:collapse.py:load_data()- Total rows loaded: 11563
INFO	PeptideCollapse:collapse.py:load_data()- Unique samples (R.FileName): 3
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing: 11563 rows remaining after phospho filter, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing complete: 11563 phospho rows retained
INFO	PeptideCollapse:collapse.py:collapse_to_sites()- Starting site-level collapse (aggregation=sum, cutoff=0.75, strategy=per_run)
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Site collapse preprocessing: 11563 rows remaining, 11563 phospho, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Per-site localization: 12722 rows got per-site prob, 0 fell back to joint prob (0.0% fallback)
INFO	PeptideCollapse

Dropped 2,625 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 8,040 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 4,189 → 4,171.
Final: 4,171 sites × 3 samples.


INFO	PeptideCollapse:collapse.py:process_complete_pipeline()- Starting complete pipeline
INFO	PeptideCollapse:collapse.py:load_data()- Loaded 28586 rows
INFO	PeptideCollapse:collapse.py:load_data()- Total rows loaded: 28586
INFO	PeptideCollapse:collapse.py:load_data()- Unique samples (R.FileName): 3
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing: 28586 rows remaining after phospho filter, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing complete: 28586 phospho rows retained
INFO	PeptideCollapse:collapse.py:collapse_to_sites()- Starting site-level collapse (aggregation=sum, cutoff=0.75, strategy=per_run)
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Site collapse preprocessing: 28586 rows remaining, 28586 phospho, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Per-site localization: 32073 rows got per-site prob, 0 fell back to joint prob (0.0% fallback)
INFO	PeptideCollapse

Dropped 3,318 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 19,590 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 10,100 → 10,064.
Final: 10,064 sites × 3 samples.


INFO	PeptideCollapse:collapse.py:process_complete_pipeline()- Starting complete pipeline
INFO	PeptideCollapse:collapse.py:load_data()- Loaded 37403 rows
INFO	PeptideCollapse:collapse.py:load_data()- Total rows loaded: 37403
INFO	PeptideCollapse:collapse.py:load_data()- Unique samples (R.FileName): 3
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing: 37403 rows remaining after phospho filter, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing complete: 37403 phospho rows retained
INFO	PeptideCollapse:collapse.py:collapse_to_sites()- Starting site-level collapse (aggregation=sum, cutoff=0.75, strategy=per_run)
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Site collapse preprocessing: 37403 rows remaining, 37403 phospho, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Per-site localization: 42335 rows got per-site prob, 0 fell back to joint prob (0.0% fallback)
INFO	PeptideCollapse

...

INFO	PeptideCollapse:collapse.py:process_complete_pipeline()- Starting complete pipeline
INFO	PeptideCollapse:collapse.py:load_data()- Loaded 68018 rows
INFO	PeptideCollapse:collapse.py:load_data()- Total rows loaded: 68018
INFO	PeptideCollapse:collapse.py:load_data()- Unique samples (R.FileName): 3
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing: 68018 rows remaining after phospho filter, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing complete: 68018 phospho rows retained
INFO	PeptideCollapse:collapse.py:collapse_to_sites()- Starting site-level collapse (aggregation=sum, cutoff=0.75, strategy=per_run)
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Site collapse preprocessing: 68018 rows remaining, 68018 phospho, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Per-site localization: 79797 rows got per-site prob, 0 fell back to joint prob (0.0% fallback)
INFO	PeptideCollapse

Dropped 5,731 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 44,374 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 20,965 → 20,899.
Final: 20,899 sites × 3 samples.


INFO	PeptideCollapse:collapse.py:process_complete_pipeline()- Starting complete pipeline
INFO	PeptideCollapse:collapse.py:load_data()- Loaded 76875 rows
INFO	PeptideCollapse:collapse.py:load_data()- Total rows loaded: 76875
INFO	PeptideCollapse:collapse.py:load_data()- Unique samples (R.FileName): 3
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing: 76875 rows remaining after phospho filter, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:preprocess_data()- Preprocessing complete: 76875 phospho rows retained
INFO	PeptideCollapse:collapse.py:collapse_to_sites()- Starting site-level collapse (aggregation=sum, cutoff=0.75, strategy=per_run)
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Site collapse preprocessing: 76875 rows remaining, 76875 phospho, 0 non-phospho removed
INFO	PeptideCollapse:collapse.py:_create_site_level_collapse()- Per-site localization: 90233 rows got per-site prob, 3 fell back to joint prob (0.0% fallback)
INFO	PeptideCollapse

Dropped 6,083 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 50,740 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 23,797 → 23,726.
Final: 23,726 sites × 3 samples.


,cells,Spec_shared_med,Spec_only_med,Spec_p,DIANN_shared_med,DIANN_only_med,DIANN_p
0,100,7.58,6.43,1.888402e-97,17.73,15.69,0.0
1,300,8.24,7.05,1.990992e-153,18.54,16.42,0.0
2,500,8.51,7.29,4.838118e-208,18.92,16.76,0.0
3,1000,8.89,7.33,0.000000e+00,19.35,17.13,0.0
4,2000,9.21,7.63,0.000000e+00,19.83,17.43,0.0
5,3000,9.53,7.91,0.000000e+00,20.33,17.80,0.0


In [22]:
from plotly.subplots import make_subplots

N_SHOW = 3000                                            # deepest input; change as desired
s_sd, d_sd = site_data_cache[N_SHOW]
Si, Di = per_site_intensity(s_sd), per_site_intensity(d_sd)
S, D = set(Si.index), set(Di.index)
shared, s_only, d_only = S & D, S - D, D - S

fig = make_subplots(rows=1, cols=2, subplot_titles=("Spectronaut intensities", "DIA-NN intensities"))
def _vio(arr, name, color, col):
    fig.add_trace(go.Violin(y=arr, name=name, line_color=color, box_visible=True,
                            meanline_visible=True, points=False, width=0.8), row=1, col=col)
_vio(Si[list(shared)].values, "shared",          "#2b8cbe", 1)
_vio(Si[list(s_only)].values, "Spectronaut-only","darkred", 1)
_vio(Di[list(shared)].values, "shared",          "#2b8cbe", 2)
_vio(Di[list(d_only)].values, "DIA-NN-only",     "#6a51a3", 2)
fig.update_layout(width=820, height=560, template="plotly_white", showlegend=False,
    title=f"Engine-unique Class I sites are lower-abundance than shared ({N_SHOW} cells)")
fig.update_yaxes(title_text="log2 intensity", row=1, col=1)
fig.update_yaxes(title_text="log2 intensity", row=1, col=2)
fig.show()
fig.write_image(str(FIG_DIR / "figure3_engineUnique_abundance.pdf"), width=820, height=560)
#fig.write_image(str(FIG_DIR / "figure3_engineUnique_abundance.png"), width=820, height=560, scale=2)
